# Explore "exxon_mobil_test_cases"

Generated for a selected volume/folder in Catalog Explorer. This notebook shows how to do the following: - viewing file metadata - filtering files by type - filtering files by size and date - processing files with AI functions - setting up Auto Loader

In [0]:
# System-filled
TARGET_PATH = "/Volumes/workspace/default/exxon_mobil_test_cases/"

## 1. List files with metadata

In [0]:
df = spark.read.format("binaryFile").load(TARGET_PATH)
display(df.select("path", "modificationTime", "length"))

## 2. Filter files by type Use `pathGlobFilter` to filter by file type.

In [0]:
df_pdfs = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.{pdf,PDF}")
    .load(TARGET_PATH)
)

display(df_pdfs.select("path", "length"))

## 3. Filter files by size and date Use `length` and `modificationTime` to filter files by size and last modified time.

In [0]:
from pyspark.sql.functions import col, current_timestamp, expr

df_files = (
    spark.read.format("binaryFile")
    .load(TARGET_PATH)
)

filtered_df = (
    df_files
    .filter(
        (col("length").between(20000, 1000000)) &
        (col("modificationTime") >= current_timestamp() - expr("INTERVAL 7 DAY"))
    )
    .orderBy(col("modificationTime").desc())
)

display(filtered_df.select("path", "length", "modificationTime"))

## 4. Process images and documents with AI functions Prerequisite: AI functions require a workspace in a supported region.

In [0]:
# Parse documents in your volume/folder
# ai_parse_document supports: PDF, JPG, JPEG, PNG, TIFF, TIF, DOC, DOCX, PPT, PPTX.
# This volume contains .xlsx files, which are NOT in the supported list.
# We read all files so the schema is properly inferred; ai_parse_document returns
# an error_status in the VARIANT for unsupported formats rather than failing.
# Escape single quotes so a path like .../o'brien stays a valid SQL string literal.
safe_path = TARGET_PATH.replace("'", "''")
result_df = spark.sql(f"""
    SELECT
      path AS file_path,
      ai_parse_document(content, map('version', '2.0')) AS parsed_content
    FROM read_files(
      '{safe_path}',
      format => 'binaryFile'
    )
    LIMIT 1
""")
display(result_df)